# SPEAR-Net — LAMES training (Colab, T4)

**SPEAR-Net**: a lightweight, color-prior-guided, recall-optimized network for
fine-grained segmentation of mining structures, with emphasis on artisanal &
small-scale (illegal-prone) mining.

This notebook trains **directly from the LAMES zip files in your Google Drive**. It
mounts Drive → finds the zips → extracts them → **auto-detects** the image/mask pairs
(no fixed layout assumed) → builds SPEAR-Net → trains → evaluates → renders CSP
explainability overlays.

> **Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*.

If you hit a bug, re-run cell 2 to pull the latest fixes, then continue.


## 1. Check GPU

In [ ]:
!nvidia-smi || echo "No GPU — set Runtime > Change runtime type > T4 GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Clone the repository

Re-run this cell to pull the latest fixes.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/prakhar443/illegal_mining.git"
BRANCH   = "spearnet-colab"
REPO_DIR = "illegal_mining"

if not os.path.exists(REPO_DIR):
    if subprocess.run(["git","clone","--branch",BRANCH,REPO_URL,REPO_DIR]).returncode != 0:
        subprocess.run(["git","clone",REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"fetch","origin",BRANCH])
    subprocess.run(["git","-C",REPO_DIR,"checkout",BRANCH])
    subprocess.run(["git","-C",REPO_DIR,"pull","origin",BRANCH])

%cd {REPO_DIR}
!git log --oneline -1


## 3. Install dependencies (~2-3 min)

In [ ]:
!pip install -q "timm>=0.9.12" "segmentation-models-pytorch>=0.3.3" \
    ptflops scikit-learn pyyaml gdown
!pip install -q -e .
print("Done. If imports fail below: Runtime > Restart session, then re-run from cell 3.")


## 4. Mount Google Drive

The LAMES zips live in **your** Drive folder
(`folders/1A27Fn7HIL8UPQlx-xUmN8Dpzxl4SQsLh`). Mounting gives the notebook direct,
authenticated access to that private folder. Click the auth prompt when it appears.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 5. Locate the dataset zips on Drive

This searches your Drive for `.zip` files so you can confirm the folder path. If your
folder has a known name, set `DRIVE_DATA_DIR` directly to skip the search.


In [ ]:
import glob, os

# Option A: set this directly if you know the path, e.g.
# DRIVE_DATA_DIR = "/content/drive/MyDrive/LAMES"
DRIVE_DATA_DIR = None

if DRIVE_DATA_DIR is None:
    hits = glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True)
    if not hits:
        hits = glob.glob("/content/drive/**/*.zip", recursive=True)
    print(f"Found {len(hits)} zip(s):")
    for h in hits[:50]:
        print("  ", h, f"({os.path.getsize(h)/1e6:.1f} MB)")
    # Most common parent directory of the zips = the dataset folder.
    if hits:
        from collections import Counter
        DRIVE_DATA_DIR = Counter(os.path.dirname(h) for h in hits).most_common(1)[0][0]

print("\nDRIVE_DATA_DIR =", DRIVE_DATA_DIR)
assert DRIVE_DATA_DIR, "Set DRIVE_DATA_DIR to the folder containing your LAMES zips."


## 6. Extract zips + auto-detect image/mask pairs

Extracts into a fast local dir (`/content/data/lames`) and prints a summary so you can
**verify** the detected pairs and mask class ids before training. If pairing looks wrong,
pass `image_hint=[...]` / `mask_hint=[...]` (folder/name keywords) to `prepare_local_dataset`.


In [ ]:
import sys; sys.path.insert(0, "src")
from spearnet.data.local import prepare_local_dataset

WORK_DIR = "/content/data/lames"
splits = prepare_local_dataset(
    source_dir=DRIVE_DATA_DIR,
    work_dir=WORK_DIR,
    zip_glob="*.zip",
    val_fraction=0.1,
    test_fraction=0.0,
    # image_hint=["images"], mask_hint=["masks"],   # uncomment to force, if needed
)


## 7. Config

Small dev subset so a run finishes in minutes. For the paper, set `subset_train = None` and `epochs = 40`.

In [ ]:
import torch
from spearnet.config import load_config

CONFIG = "configs/spearnet_10class.yaml"   # or spearnet_3class.yaml / spearnet_binary.yaml
cfg = load_config(CONFIG)

cfg.data.source     = "local"
cfg.data.local_root = WORK_DIR
# Colab-friendly dev settings (comment out for the full paper run)
cfg.data.subset_train = 1500
cfg.data.subset_val   = 400
cfg.optim.epochs      = 8
cfg.data.num_workers  = 2
cfg.run.device        = "cuda" if torch.cuda.is_available() else "cpu"

from spearnet.utils import set_seed
set_seed(cfg.run.seed)
print(f"Task={cfg.data.task} ({cfg.num_classes} classes) | csp={cfg.model.csp_mode} | "
      f"backbone={cfg.model.backbone} | device={cfg.run.device}")


## 8. Build dataloaders

In [ ]:
from spearnet.data import build_dataloaders
loaders = build_dataloaders(cfg, splits=("train", "val"))
print("train batches:", len(loaders["train"]), "| val batches:", len(loaders["val"]))
batch = next(iter(loaders["train"]))
print({k: tuple(v.shape) for k, v in batch.items()})
print("mask classes in batch:", torch.unique(batch["mask"]).tolist())


## 9. Visualize a sample + the Color-Spectral Prior

ExG (greenness), Brightness, Redness (iron-oxide), RGB-VI — computed on the fly.

In [ ]:
import matplotlib.pyplot as plt
from spearnet.data import compute_csp_priors
from spearnet.data.priors import CSP_PRIOR_NAMES
from spearnet.utils.viz import colorize_mask

img = batch["image"][0]
priors = compute_csp_priors(img.unsqueeze(0))[0]
fig, ax = plt.subplots(1, 6, figsize=(22, 4))
ax[0].imshow(img.permute(1,2,0).numpy()); ax[0].set_title("RGB")
ax[1].imshow(colorize_mask(batch["mask"][0].numpy())); ax[1].set_title("Mask")
for i, name in enumerate(CSP_PRIOR_NAMES):
    ax[i+2].imshow(priors[i].numpy(), cmap="viridis"); ax[i+2].set_title(name)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## 10. Build SPEAR-Net + efficiency report

Params / GFLOPs / latency / peak VRAM — backs the deployability claims.

In [ ]:
from spearnet.models import build_model
from spearnet.utils import count_parameters, measure_efficiency
import json

model = build_model(cfg)
print("Parameters:", count_parameters(model))
print(json.dumps(measure_efficiency(model, cfg.data.image_size, cfg.run.device), indent=2))


## 11. Train

In [ ]:
from spearnet.engine import Trainer
trainer = Trainer(model, loaders, cfg)
summary = trainer.train()
print("best", cfg.run.save_best_metric, "=", summary["best_metric"])


## 12. Evaluate (per-class IoU + area-stratified recall)

In [ ]:
from spearnet.engine import evaluate
device = torch.device(cfg.run.device)
results = evaluate(model, loaders["val"], cfg, device, compute_area_stratified=True)
print(f"mIoU={results['miou']:.4f}  mF1={results['mean_f1']:.4f}  "
      f"mRecall={results['mean_recall']:.4f}  pixAcc={results['pixel_acc']:.4f}")
print("\nPer-class IoU / recall:")
for cls, m in results["per_class"].items():
    print(f"  {cls:>18}: IoU={m['iou']:.3f}  recall={m['recall']:.3f}  support={m['support']}")
print("\nArea-stratified recall:", results.get("area_stratified_recall"))


## 13. Explainability — predictions + CSP attention overlay

In [ ]:
from spearnet.utils import save_prediction_panel
model.eval()
with torch.no_grad():
    vb = next(iter(loaders["val"]))
    out = model(vb["image"].to(device))
    preds = out["logits"].argmax(1).cpu()
    attn = out.get("attn")

os.makedirs("figures", exist_ok=True)
for i in range(min(3, preds.shape[0])):
    a = attn[i].cpu() if attn is not None else None
    save_prediction_panel(vb["image"][i], vb["mask"][i], preds[i],
                          f"figures/panel_{i}.png", attn=a, class_names=cfg.class_names)
from IPython.display import Image as IPImage, display
for i in range(min(3, preds.shape[0])):
    display(IPImage(f"figures/panel_{i}.png"))


## 14. Next steps — ablations, baselines & the full run

```python
# Full paper run (long): full data + 40 epochs
!python scripts/train.py --config configs/spearnet_10class.yaml \
    --set data.source=local data.local_root=/content/data/lames \
          data.subset_train=null optim.epochs=40

# Ablation table (compare runs/*/history.json)
for c in ["ablation_1_baseline","ablation_2_recall_loss",
          "ablation_3_csp_concat","ablation_4_full_spearnet"]:
    !python scripts/train.py --config configs/{c}.yaml \
        --set data.source=local data.local_root=/content/data/lames \
              optim.epochs=8 data.subset_train=1500

# Baselines (U-Net / Attention U-Net / DeepLabV3+ / U-Net++)
!python scripts/train.py --config configs/baseline_unet.yaml \
    --set data.source=local data.local_root=/content/data/lames model.name=deeplabv3p
```

Switch `CONFIG` to `configs/spearnet_3class.yaml` for the **ASM-vs-LSM** headline result,
or `configs/spearnet_binary.yaml` for the mining detector.
